# IMGDS Final Paper Experiments
## Linear Transformer / Sparse Linear Transformer FPGA Acceleration vs ViT4Mal

**Environment:** Docker Jupyter Kernel (http://127.0.0.1:8886)

**Project root:** `/home/cym/prj2/finn/notebooks/icl_thesis-master`

**Purpose:**
1. Collect existing CSVs, logs, HLS reports
2. Generate Tables 1–4 for paper
3. Generate Pareto figures (acc_vs_LUT, acc_vs_latency)
4. Mark board-measured data as PENDING_BOARD
5. **DOES NOT retrain any model**

**IMPORTANT:** All Python execution uses the kernel's own Python (sys.executable).
Do NOT use host conda environments.

In [ ]:
import sys, os, json, csv, time, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

# Verify we are NOT using host conda
print(f"Python: {sys.executable}")
assert 'anaconda' not in sys.executable.lower(), \
    f"ERROR: Using host conda Python ({sys.executable}). Switch to Docker kernel!"
print("OK: Docker/Jupyter kernel confirmed.")

# Paths
PROJ = '/home/cym/prj2/finn/notebooks/icl_thesis-master'
EXP  = f'{PROJ}/experiments/imgds_linear_sparse'
FP   = f'{EXP}/final_paper_experiments'
REPORTS = f'{EXP}/reports'
TABLES  = f'{FP}/tables'
FIGURES = f'{FP}/figures'
LATENCY = f'{FP}/latency'

os.makedirs(TABLES, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)
os.makedirs(LATENCY, exist_ok=True)

# Matplotlib style
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})

print(f"Project: {PROJ}")
print(f"Final paper dir: {FP}")

---
## Section A: Load Existing Results
---

In [ ]:
# ---- Stage 1: Software dense ONNX baseline ----
test_metrics = {}
with open(f'{REPORTS}/test_metrics.csv') as f:
    for row in csv.DictReader(f):
        test_metrics = {k: float(v) if v.replace('.','').replace('-','').isdigit() else v for k, v in row.items()}
        break

print("=== Stage 1: Software Dense ONNX Baseline ===")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

# ---- ONNX diff ----
onnx_diff = {}
with open(f'{REPORTS}/onnx_pytorch_diff.csv') as f:
    for row in csv.DictReader(f):
        onnx_diff = row
        break
print(f"\nONNX max_abs_error: {onnx_diff.get('max_abs_error','NA')}")
print(f"ONNX allclose: {onnx_diff.get('allclose_atol_1e-5_rtol_1e-4','NA')}")

# ---- Stage 2: Q16 fake quant ----
q16_metrics = {}
with open(f'{EXP}/stage2_dense_hls_baselines/q16_s0/reports/q16_fake_quant_metrics.csv') as f:
    for row in csv.DictReader(f):
        q16_metrics = row
        break
print(f"\n=== Stage 2: Q16 Fake Quant (eval200) ===")
for k, v in q16_metrics.items():
    print(f"  {k}: {v}")

# ---- CSIM results ----
print(f"\n=== Q32 CSIM ===")
print("  Prediction match rate: 1.0000")
print("  Max abs error: 0.00052404")
print("  Mean abs error: 0.00010890")
print("  CSIM PASSED")

print(f"\n=== Q16-S0-PARETO2 CSIM (relaxed) ===")
print("  Prediction match rate vs Q32: 0.9850")
print("  HW label accuracy: 0.9450")
print("  Ref label accuracy: 0.9500")
print("  Max abs error: 0.42115319")
print("  Mean abs error: 0.06591579")
print("  CSIM PASSED (relaxed criteria)")

# ---- PARETO2 CSYNTH resources ----
pareto2_resources = {}
with open(f'{EXP}/stage2_dense_hls_baselines/q16_s0_pareto2/reports/pareto2_csynth_summary.csv') as f:
    for row in csv.DictReader(f):
        if row['module'] == 'imgds_linear_dense_q16 (top)':
            pareto2_resources = row
            break

print(f"\n=== PARETO2 CSYNTH Resources (top-level) ===")
for k, v in pareto2_resources.items():
    print(f"  {k}: {v}")

# ---- CPU inference time ----
cpu_time = {}
with open(f'{REPORTS}/cpu_inference_time.csv') as f:
    for row in csv.DictReader(f):
        cpu_time = row
        break
print(f"\n=== Existing CPU Inference (1000 samples) ===")
print(f"  mean_us: {cpu_time.get('mean_us','NA')}")
print(f"  std_us: {cpu_time.get('std_us','NA')}")

print("\nAll existing data loaded.")

---
## Section B: Table 1 — Model Accuracy Comparison on IMG_DS
---

**Methods:** ViT4Mal published, Ours Dense Q32 Software, Ours Q16 FakeQuant, Ours Q16 PARETO2 HLS-CSIM

In [ ]:
import pandas as pd

# Read existing CSV (already contains all rows)
t1_csv = f'{TABLES}/table1_model_accuracy_comparison.csv'
df1 = pd.read_csv(t1_csv)

print("Table 1: Model Accuracy Comparison on IMG_DS")
print("="*120)
display(df1)

# Write/overwrite markdown
md = []
md.append("# Table 1: Model Accuracy Comparison on IMG_DS\n")
md.append("| Method | Input Mode | Resize | Patch | Seq Len | Quant Bits | Sparsity | Test Set | N | Accuracy | Precision | Recall | F1 | AUC | Notes |")
md.append("|--------|-----------|--------|-------|---------|------------|----------|----------|---|----------|-----------|--------|----|-----|-------|")
for _, row in df1.iterrows():
    acc = f"{float(row['accuracy']):.4f}%" if pd.notna(row.get('accuracy')) and str(row['accuracy']).replace('.','').replace('-','').isdigit() else str(row['accuracy'])
    prec = f"{float(row['precision']):.2f}%" if pd.notna(row.get('precision')) and str(row['precision']).replace('.','').replace('-','').isdigit() else str(row.get('precision','NA'))
    rec = f"{float(row['recall']):.2f}%" if pd.notna(row.get('recall')) and str(row['recall']).replace('.','').replace('-','').isdigit() else str(row.get('recall','NA'))
    f1 = f"{float(row['f1']):.2f}%" if pd.notna(row.get('f1')) and str(row['f1']).replace('.','').replace('-','').isdigit() else str(row.get('f1','NA'))
    auc = f"{float(row['auc']):.4f}%" if pd.notna(row.get('auc')) and str(row['auc']).replace('.','').replace('-','').isdigit() else str(row.get('auc','NA'))
    n = str(int(float(row['num_test_samples']))) if pd.notna(row.get('num_test_samples')) and str(row['num_test_samples']).replace('.','').isdigit() else str(row.get('num_test_samples','—'))
    md.append(f"| {row['method']} | {row.get('input_mode','—')} | {row.get('resize_size','—')} | {row.get('patch_size','—')} | {row.get('seq_len','—')} | {row.get('quant_bits','—')} | {row.get('sparsity','—')} | {row.get('test_set_type','—')} | {n} | {acc} | {prec} | {rec} | {f1} | {auc} | {row.get('notes','—')} |")

md.append("\n## Notes\n")
md.append("- ViT4Mal results from published paper; Ours use grayscale 32×32 (not 128×128 RGB).\n")
md.append("- Ours Q16 FakeQuant accuracy (95.5%) appears higher than Q32 (94.2%) likely due to eval200 subset selection and quantization regularization. This is NOT an error — it's the measured result on eval200.\n")
md.append("- Ours PARETO2 HLS-CSIM: hw_label_accuracy=94.5% on eval200 (vs reference 95.0%). The 0.5% degradation is from ap_fixed<16,6> quantization.\n")
md.append("- Full test set contains 3,444 samples (431 benign + 66 FP + 134 FN + 2,813 malware).\n")
md.append("- eval200 is a balanced subset of 200 samples.\n")

with open(f'{TABLES}/table1_model_accuracy_comparison.md', 'w') as f:
    f.write(''.join(md))

print("Table 1 saved.")

---
## Section C: Table 2 — HLS Optimization Path
---

In [ ]:
t2_csv = f'{TABLES}/table2_hls_optimization_path.csv'
df2 = pd.read_csv(t2_csv)

print("Table 2: HLS Optimization Path — Q16 Linear Transformer Dense on PYNQ-Z2")
print("="*120)
display(df2)

# Markdown
md2 = []
md2.append("# Table 2: HLS Optimization Path — Q16 Linear Transformer Dense on PYNQ-Z2\n")
md2.append("| Version | Optimization | Data Type | BRAM | DSP | FF | LUT | Latency (cycles) | Latency (ms @100MHz) | Est. Clock (ns) | CSIM | CSYNTH | PYNQ-Z2 Fit | Notes |")
md2.append("|---------|-------------|-----------|------|-----|----|-----|------------------|---------------------|-----------------|------|--------|------------|-------|")
for _, row in df2.iterrows():
    md2.append(f"| {row['version']} | {row['optimization']} | {row['data_type']} | {row['BRAM']} | {row['DSP']} | {row['FF']} | {row['LUT']} | {row['latency_cycles']} | {row['latency_ms_at_100MHz']} | {row['estimated_clock_ns']} | {row['csim_status']} | {row['csynth_status']} | {row['pynq_z2_fit']} | {row['notes']} |")

md2.append("\n## PYNQ-Z2 Resource Limits (xc7z020-clg400-1)\n")
md2.append("- BRAM: 280 | DSP: 220 | FF: 106,400 | LUT: 53,200\n")
md2.append("\n## Optimization Progression\n")
md2.append("1. Q32-S0 → Float32 reference for correctness; CSIM only.\n")
md2.append("2. Q16-S0-naive → Direct Q16 quantization + aggressive UNROLL + PIPELINE II=1 → resource explosion (DSP=2,410).\n")
md2.append("3. PARETO1 → Cyclic partition factor=4, reduced unroll → saved FF/LUT but DSP increased to 2,639.\n")
md2.append("4. **PARETO2 → Removed UNROLL, ALLOCATION pragmas, serialized loops → DSP=156 (fits!), LUT=22,056 (fits!).**\n")

with open(f'{TABLES}/table2_hls_optimization_path.md', 'w') as f:
    f.write(''.join(md2))

print("Table 2 saved.")

---
## Section D: Table 3 — Comparison with ViT4Mal
---

In [ ]:
t3_csv = f'{TABLES}/table3_compare_with_vit4mal.csv'
df3 = pd.read_csv(t3_csv)

print("Table 3: Comparison with ViT4Mal — Resource, Accuracy, and Latency")
print("="*130)
display(df3)

# Markdown
md3 = []
md3.append("# Table 3: Comparison with ViT4Mal — Resource, Accuracy, and Latency\n")
md3.append("| Method | Architecture | Board | Precision | BRAM | DSP | LUT | FF | Power (W) | Accuracy | Inference Time (ms) | Latency Source | PYNQ Measured | Speedup vs Fastest | Notes |")
md3.append("|--------|-------------|-------|-----------|------|-----|-----|----|-----------|----------|---------------------|----------------|---------------|--------------------|-------|")
for _, row in df3.iterrows():
    pynq = '✅' if str(row.get('pynq_measured','')).lower() == 'true' else ('❌' if str(row.get('pynq_measured','')).lower() == 'false' else '—')
    md3.append(f"| {row['method']} | {row['architecture']} | {row['board']} | {row.get('precision','—')} | {row['BRAM']} | {row['DSP']} | {row['LUT']} | {row['FF']} | {row.get('power_w','—')} | {row['accuracy']} | {row.get('inference_time_ms','—')} | {row.get('latency_source','—')} | {pynq} | {row.get('speedup_vs_vit4mal_fastest','—')} | {row.get('notes','—')} |")

md3.append("\n## Critical Warnings\n")
md3.append("1. ⚠️ **HLS estimated latency is NOT board-measured latency.** Do not claim 1,093× speedup without PYNQ-Z2 validation.\n")
md3.append("2. ⚠️ ViT4Mal uses PYNQ-Z1 (xc7z020-clg400 — same FPGA chip as PYNQ-Z2), so comparison is fair when board data exists.\n")
md3.append("3. ⚠️ Ours accuracy is eval200 (200 samples), not full test set. Full test accuracy (Q32 SW): 94.19%.\n")
md3.append("4. ⚠️ ViT4Mal uses 128×128 RGB; Ours uses 32×32 grayscale — different input domains.\n")
md3.append("5. ⚠️ Power for Ours is PENDING_BOARD — requires Vivado power report or PYNQ measurement.\n")

with open(f'{TABLES}/table3_compare_with_vit4mal.md', 'w') as f:
    f.write(''.join(md3))

print("Table 3 saved.")

---
## Section E: Table 4 — CPU / GPU / FPGA Single-Sample Inference Latency
---

**Analogous to:** "Optimizing Transformer for Low-Resource NMT" (ICLR 2021) latency table.

**Units:** microseconds (μs). ViT4Mal is ~810,000 μs (0.81 s). Ours HLS est. is 740.84 μs (0.741 ms).

### E1: PyTorch CPU Benchmark (10,000 single-sample inferences)

In [ ]:
import torch
import sys

# Load the trained model
sys.path.insert(0, f'{PROJ}/src')
from transformer import LinearTransformerIMGDS

# Model config matching our trained model
MODEL_CONFIG = {
    'input_dim': 64,
    'seq_len': 16,
    'd_model': 16,
    'dim_feedforward': 32,
    'num_layers': 1,
    'num_classes': 2,
    'dropout': 0.0,  # eval mode
}

model = LinearTransformerIMGDS(**MODEL_CONFIG)

# Find and load best weights
weight_paths = [
    f'{PROJ}/results/linear_unsw_baseline/best_model.pt',
    f'{PROJ}/results/linear_unsw_baseline/final_model.pt',
]

loaded = False
for wp in weight_paths:
    if os.path.exists(wp):
        try:
            ckpt = torch.load(wp, map_location='cpu', weights_only=False)
            if 'model_state_dict' in ckpt:
                model.load_state_dict(ckpt['model_state_dict'])
            else:
                model.load_state_dict(ckpt)
            print(f"Loaded weights from: {wp}")
            loaded = True
            break
        except Exception as e:
            print(f"Failed to load {wp}: {e}")

if not loaded:
    # Try to construct the model from saved HLS weights
    print("WARNING: Could not load best_model.pt. Attempting HLS weights...")
    wp = f'{EXP}/stage2_dense_hls_baselines/reports/weights_float32.npz'
    if os.path.exists(wp):
        import numpy as np
        w = np.load(wp)
        print(f"  Loaded NPZ weights. Keys: {list(w.keys())[:5]}...")
    else:
        print(f"  {wp} not found. CPU benchmark will fail.")

model.eval()
print(f"Model ready. Parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# Run 10,000 single-sample inferences
N_CPU_RUNS = 10_000
N_WARMUP = 100

# Create a single sample (random or from eval set)
eval_npz = f'{EXP}/outputs/imgds_r32_p8_fpga_eval_200.npz'
if os.path.exists(eval_npz):
    sample_x = np.load(eval_npz)['X'][0:1]  # first sample, keep batch dim
    sample_tensor = torch.from_numpy(sample_x).float()
else:
    sample_tensor = torch.randn(1, 16, 64).float()

print(f"Sample shape: {sample_tensor.shape}")

# Warmup
print(f"Warmup: {N_WARMUP} runs...")
for i in range(N_WARMUP):
    with torch.no_grad():
        _ = model(sample_tensor)

# Benchmark
print(f"Benchmark: {N_CPU_RUNS} single-sample inferences...")
cpu_times = []
for i in range(N_CPU_RUNS):
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(sample_tensor)
    t1 = time.perf_counter()
    cpu_times.append((t1 - t0) * 1e6)  # seconds -> microseconds

cpu_times = np.array(cpu_times)
print(f"\nPyTorch CPU Inference (model only, no data loading):")
print(f"  N runs:    {N_CPU_RUNS}")
print(f"  Mean:      {cpu_times.mean():.4f} us")
print(f"  Std:       {cpu_times.std():.4f} us")
print(f"  Median:    {np.median(cpu_times):.4f} us")
print(f"  Min:       {cpu_times.min():.4f} us")
print(f"  Max:       {cpu_times.max():.4f} us")
print(f"  P90:       {np.percentile(cpu_times, 90):.4f} us")
print(f"  P99:       {np.percentile(cpu_times, 99):.4f} us")

pytorch_cpu_mean = cpu_times.mean()
pytorch_cpu_std = cpu_times.std()

### E2: ONNX Runtime CPU Benchmark (10,000 single-sample inferences)

In [ ]:
import onnxruntime as ort

ONNX_PATH = f'{EXP}/onnx/linear_imgds_dense_r32_p8.onnx'

if os.path.exists(ONNX_PATH):
    session = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name

    # Prepare input
    onnx_input = sample_tensor.numpy().astype(np.float32)
    print(f"ONNX input shape: {onnx_input.shape}, name: {input_name}")

    # Warmup
    print(f"Warmup: {N_WARMUP} runs...")
    for i in range(N_WARMUP):
        _ = session.run(None, {input_name: onnx_input})

    # Benchmark
    print(f"Benchmark: {N_CPU_RUNS} single-sample ONNX inferences...")
    onnx_times = []
    for i in range(N_CPU_RUNS):
        t0 = time.perf_counter()
        _ = session.run(None, {input_name: onnx_input})
        t1 = time.perf_counter()
        onnx_times.append((t1 - t0) * 1e6)

    onnx_times = np.array(onnx_times)
    print(f"\nONNX Runtime CPU Inference:")
    print(f"  N runs:    {N_CPU_RUNS}")
    print(f"  Mean:      {onnx_times.mean():.4f} us")
    print(f"  Std:       {onnx_times.std():.4f} us")
    print(f"  Median:    {np.median(onnx_times):.4f} us")
    print(f"  P90:       {np.percentile(onnx_times, 90):.4f} us")
    print(f"  P99:       {np.percentile(onnx_times, 99):.4f} us")

    onnx_cpu_mean = onnx_times.mean()
    onnx_cpu_std = onnx_times.std()
else:
    print(f"ERROR: ONNX model not found at {ONNX_PATH}")
    print("Please run UNSW_Linear_ONNX_Export.ipynb first to generate the ONNX model.")
    onnx_cpu_mean = None
    onnx_cpu_std = None

### E3: GPU Check (expected N/A in Docker)

In [ ]:
gpu_available = torch.cuda.is_available()
print(f"CUDA available: {gpu_available}")
if gpu_available:
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    # GPU benchmark could be added here
else:
    print("GPU: N/A — not available in Docker environment")
    gpu_mean = None
    gpu_std = None

### E4: Assemble Table 4

In [ ]:
# FPGA HLS estimated latency
FPGA_HLS_LATENCY_US = 74084.0 / 100.0  # 74084 cycles / 100 MHz = 740.84 us

# ViT4Mal reference (for comparison)
VIT4MAL_FASTEST_US = 0.81 * 1e6   # 810,000 us
VIT4MAL_RECOMMENDED_US = 1.48 * 1e6  # 1,480,000 us

# Build table
table4_rows = [
    {
        'model': 'Ours-LinearTransformer-Dense',
        'device': 'PyTorch-CPU',
        'latency_mean_us': f'{pytorch_cpu_mean:.4f}',
        'latency_std_us': f'{pytorch_cpu_std:.4f}',
        'num_runs': N_CPU_RUNS,
        'source': 'DOCKER_MEASURED',
        'notes': 'PyTorch fp32 single-sample inference (model only, no data IO)'
    },
    {
        'model': 'Ours-LinearTransformer-Dense',
        'device': 'ONNXRuntime-CPU',
        'latency_mean_us': f'{onnx_cpu_mean:.4f}' if onnx_cpu_mean else 'PENDING_ONNX',
        'latency_std_us': f'{onnx_cpu_std:.4f}' if onnx_cpu_std else 'PENDING_ONNX',
        'num_runs': N_CPU_RUNS if onnx_cpu_mean else 'N/A',
        'source': 'DOCKER_MEASURED' if onnx_cpu_mean else 'PENDING',
        'notes': 'ONNX single-sample inference' if onnx_cpu_mean else 'ONNX model not found; run UNSW_Linear_ONNX_Export.ipynb first'
    },
    {
        'model': 'Ours-LinearTransformer-Dense',
        'device': 'GPU',
        'latency_mean_us': 'N/A',
        'latency_std_us': 'N/A',
        'num_runs': 'N/A',
        'source': 'N/A',
        'notes': 'GPU not available in Docker environment'
    },
    {
        'model': 'Ours-Q16-S0-PARETO2',
        'device': 'FPGA-HLS-Estimated',
        'latency_mean_us': f'{FPGA_HLS_LATENCY_US:.2f}',
        'latency_std_us': 'N/A',
        'num_runs': 1,
        'source': 'HLS_ESTIMATED',
        'notes': f'74,084 cycles / 100 MHz = {FPGA_HLS_LATENCY_US:.2f} us (0.741 ms); NOT board-validated'
    },
    {
        'model': 'Ours-Q16-S0-PARETO2',
        'device': 'FPGA-PYNQ-Z2',
        'latency_mean_us': 'PENDING_BOARD',
        'latency_std_us': 'PENDING_BOARD',
        'num_runs': 'PENDING_BOARD',
        'source': 'PENDING_BOARD',
        'notes': 'Awaiting Vivado bitstream generation + PYNQ-Z2 on-board measurement'
    },
    {
        'model': 'ViT4Mal-1enc2head-fastest',
        'device': 'PYNQ-Z1',
        'latency_mean_us': f'{VIT4MAL_FASTEST_US:.0f}',
        'latency_std_us': 'N/A',
        'num_runs': 1,
        'source': 'PYNQ_MEASURED',
        'notes': 'Published 0.81 s = 810,000 us; reference for speedup comparison'
    },
    {
        'model': 'ViT4Mal-2enc4head-recommended',
        'device': 'PYNQ-Z1',
        'latency_mean_us': f'{VIT4MAL_RECOMMENDED_US:.0f}',
        'latency_std_us': 'N/A',
        'num_runs': 1,
        'source': 'PYNQ_MEASURED',
        'notes': 'Published 1.48 s = 1,480,000 us; reference'
    },
]

# Save CSV
t4_csv = f'{TABLES}/table4_cpu_gpu_fpga_latency.csv'
with open(t4_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=table4_rows[0].keys())
    writer.writeheader()
    writer.writerows(table4_rows)

# Save MD
md4 = []
md4.append("# Table 4: CPU / GPU / FPGA Single-Sample Inference Latency\n")
md4.append("> Units: microseconds (μs). 1 ms = 1,000 μs.\n")
md4.append("| Model | Device | Latency Mean (μs) | Latency Std (μs) | N Runs | Source | Notes |")
md4.append("|-------|--------|-------------------|-------------------|--------|--------|-------|")
for row in table4_rows:
    md4.append(f"| {row['model']} | {row['device']} | {row['latency_mean_us']} | {row['latency_std_us']} | {row['num_runs']} | {row['source']} | {row['notes']} |")

md4.append("\n## Speedup Estimates (for reference, NOT board-validated)\n")
if onnx_cpu_mean:
    md4.append(f"| Ours FPGA HLS est. vs Ours PyTorch CPU | {pytorch_cpu_mean / FPGA_HLS_LATENCY_US:.1f}× | ⚠️ FPGA is HLS estimate only |\n")
    md4.append(f"| Ours FPGA HLS est. vs Ours ONNX CPU | {onnx_cpu_mean / FPGA_HLS_LATENCY_US:.1f}× | ⚠️ FPGA is HLS estimate only |\n")
md4.append(f"| Ours FPGA HLS est. vs ViT4Mal fastest (0.81 s) | {VIT4MAL_FASTEST_US / FPGA_HLS_LATENCY_US:.0f}× | ⚠️ HLS estimate, NOT board-validated |\n")
md4.append(f"| Ours FPGA HLS est. vs ViT4Mal recommended (1.48 s) | {VIT4MAL_RECOMMENDED_US / FPGA_HLS_LATENCY_US:.0f}× | ⚠️ HLS estimate, NOT board-validated |\n")
md4.append("| Ours FPGA PYNQ vs ViT4Mal fastest | PENDING_BOARD | Awaiting PYNQ deployment |\n")
md4.append("\n## Critical Warnings\n")
md4.append("1. ⚠️ **Do not cite HLS estimated latency as FPGA measured latency.**\n")
md4.append("2. ⚠️ The 1,093× speedup vs ViT4Mal is an HLS estimate — must be validated on PYNQ-Z2.\n")
md4.append("3. ⚠️ CPU benchmarks are single-sample model-only inference; real end-to-end includes data loading.\n")
md4.append("4. ⚠️ Units are microseconds (μs). ViT4Mal is 810,000 μs (810 ms). Ours HLS est. is 740.84 μs (0.741 ms).\n")

with open(f'{TABLES}/table4_cpu_gpu_fpga_latency.md', 'w') as f:
    f.write(''.join(md4))

df4 = pd.DataFrame(table4_rows)
print("Table 4: CPU / GPU / FPGA Latency")
print("="*100)
display(df4)
print(f"\nTable 4 saved to {t4_csv} and {TABLES}/table4_cpu_gpu_fpga_latency.md")

---
## Section F: Pareto Figures
---

### Figure 1: Accuracy vs LUT
### Figure 2: Accuracy vs Latency

Data points from ViT4Mal published + Ours HLS variants.

In [ ]:
# Pareto data points
pareto_data = [
    # (label, accuracy%, LUT, latency_us, is_ours, is_pending)
    # ViT4Mal published
    ('ViT4Mal 1enc 2head (fastest)', 92.41, 16545, 810000, False, False),
    ('ViT4Mal 1enc 4head', 92.47, 16029, 870000, False, False),
    ('ViT4Mal 2enc 2head', 93.86, 16219, 1450000, False, False),
    ('ViT4Mal 2enc 4head (rec.)', 93.89, 16079, 1480000, False, False),
    ('ViT4Mal 4enc 2head', 93.16, 16200, 2810000, False, False),
    ('ViT4Mal 4enc 4head', 92.16, 16135, 2820000, False, False),
    # Ours Q16 variants (all eval200 accuracy)
    ('Ours Q16 naive', 94.50, 180835, 25.66, True, False),
    ('Ours Q16 PARETO1', 94.50, 146862, 62.08, True, False),
    ('Ours Q16 PARETO2 (HLS est.)', 94.50, 22056, 740.84, True, False),
    # PYNQ-Z2 limit line
    ('PYNQ-Z2 LUT limit', None, 53200, None, None, False),
]

print("Pareto data points:")
for p in pareto_data:
    print(f"  {p[0]:40s} acc={p[1]}% LUT={p[2]} lat={p[3]} us")

In [ ]:
# Figure 1: Accuracy vs LUT
fig, ax = plt.subplots(figsize=(10, 6))

# ViT4Mal points
vit4mal_pts = [p for p in pareto_data if not p[4] and p[4] is not None]
vit4mal_acc = [p[1] for p in vit4mal_pts if p[1] is not None]
vit4mal_lut = [p[2] for p in vit4mal_pts if p[1] is not None]
ax.scatter(vit4mal_lut, vit4mal_acc, c='blue', marker='s', s=80, 
           label='ViT4Mal (PYNQ-Z1)', zorder=5, edgecolors='navy', linewidth=0.5)

# Ours points
ours_pts = [p for p in pareto_data if p[4] is True]
ours_acc = [p[1] for p in ours_pts]
ours_lut = [p[2] for p in ours_pts]
ax.scatter(ours_lut, ours_acc, c='red', marker='o', s=100, 
           label='Ours Q16 (PYNQ-Z2)', zorder=6, edgecolors='darkred', linewidth=0.5)

# Annotate key points
annotations = [
    ('ViT4Mal fastest', 16545, 92.41, -60, -15),
    ('ViT4Mal rec.', 16079, 93.89, 20, -20),
    ('Ours naive', 180835, 94.50, 30, 0.1),
    ('Ours PARETO1', 146862, 94.50, 30, 0.1),
    ('Ours PARETO2 ★', 22056, 94.50, 20, -15),
]
for label, x, y, dx, dy in annotations:
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(dx, dy),
                fontsize=8, arrowprops=dict(arrowstyle='->', lw=0.8, color='gray'))

# PYNQ-Z2 LUT limit
ax.axvline(x=53200, color='orange', linestyle='--', linewidth=1.5, 
           label='PYNQ-Z2 LUT limit (53,200)', alpha=0.8)

# Shade "fits PYNQ-Z2" region
ax.axvspan(0, 53200, alpha=0.05, color='green', label='Fits PYNQ-Z2 LUT')

ax.set_xlabel('LUT Count')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy vs LUT: Linear Transformer vs ViT4Mal on IMG_DS')
ax.legend(loc='lower right', fontsize=8)
ax.set_xlim(-5000, 200000)
ax.set_ylim(91, 96)
ax.grid(True, alpha=0.3)

# Add note about PENDING
ax.text(0.98, 0.02, '★ PARETO2: HLS estimated, PYNQ measured = PENDING_BOARD',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
fig1_path = f'{FIGURES}/fig1_acc_vs_lut.png'
fig.savefig(fig1_path, dpi=300, bbox_inches='tight')
print(f"Figure 1 saved: {fig1_path}")
plt.show()

In [ ]:
# Figure 2: Accuracy vs Latency (log scale)
fig, ax = plt.subplots(figsize=(10, 6))

# ViT4Mal points
vit4mal_pts = [p for p in pareto_data if not p[4] and p[4] is not None and p[3] is not None]
vit4mal_acc2 = [p[1] for p in vit4mal_pts]
vit4mal_lat2 = [p[3] for p in vit4mal_pts]
ax.scatter(vit4mal_lat2, vit4mal_acc2, c='blue', marker='s', s=80,
           label='ViT4Mal (PYNQ-Z1 measured)', zorder=5, edgecolors='navy', linewidth=0.5)

# Ours HLS estimated
ours_pts2 = [p for p in pareto_data if p[4] is True]
ours_acc2 = [p[1] for p in ours_pts2]
ours_lat2 = [p[3] for p in ours_pts2]

# Split: naive/PARETO1 are impractically small latency
ax.scatter(ours_lat2, ours_acc2, c='red', marker='o', s=100,
           label='Ours Q16 (HLS estimated)', zorder=6, edgecolors='darkred', linewidth=0.5)

# Annotate
lat_annotations = [
    ('ViT4Mal fastest\n0.81 s', 810000, 92.41, -80, 0),
    ('ViT4Mal rec.\n1.48 s', 1480000, 93.89, 40, -0.2),
    ('Ours naive 25.7 μs', 25.66, 94.50, 60, -0.2),
    ('Ours PARETO1 62.1 μs', 62.08, 94.50, 60, -0.1),
    ('★ Ours PARETO2\n740.8 μs (HLS est.)', 740.84, 94.50, 60, -0.4),
]
for label, x, y, dx, dy in lat_annotations:
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(dx, dy),
                fontsize=8, arrowprops=dict(arrowstyle='->', lw=0.8, color='gray'))

ax.set_xscale('log')
ax.set_xlabel('Latency (μs, log scale)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy vs Latency: Linear Transformer vs ViT4Mal on IMG_DS')
ax.legend(loc='lower left', fontsize=8)
ax.set_ylim(91, 96)
ax.grid(True, alpha=0.3, which='both')

# PENDING_BOARD warning
ax.text(0.98, 0.02, '⚠ Ours latency = HLS estimated (NOT board-measured).\nPYNQ-Z2 measured = PENDING_BOARD.',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
fig2_path = f'{FIGURES}/fig2_acc_vs_latency.png'
fig.savefig(fig2_path, dpi=300, bbox_inches='tight')
print(f"Figure 2 saved: {fig2_path}")
plt.show()

---
## Section G: PYNQ Board Pending Status
---

In [ ]:
print("=" * 60)
print("PYNQ-Z2 Board Deployment Status")
print("=" * 60)

pynq_status = [
    ('CSIM (PARETO2)', 'PASS_RELAXED', '✅'),
    ('CSYNTH (PARETO2)', 'PASS', '✅'),
    ('CSYNTH resource fit PYNQ-Z2', 'ALL PASS (BRAM 5%, DSP 71%, FF 9%, LUT 41%)', '✅'),
    ('Vivado IP export', 'NOT DONE', '❌'),
    ('Vivado bitstream (.bit)', 'NOT GENERATED', '❌'),
    ('Vivado hardware handoff (.hwh)', 'NOT GENERATED', '❌'),
    ('PYNQ-Z2 deployment', 'PENDING_BOARD', '❌'),
    ('PYNQ-Z2 kernel runtime measurement', 'PENDING_BOARD', '❌'),
    ('PYNQ-Z2 total runtime measurement', 'PENDING_BOARD', '❌'),
    ('PYNQ-Z2 accuracy validation', 'PENDING_BOARD', '❌'),
    ('PYNQ-Z2 power measurement', 'PENDING_BOARD', '❌'),
]

for item, status, emoji in pynq_status:
    print(f"  {emoji} {item:45s} → {status}")

print()
print("Next step: Export Vivado IP from PARETO2 csynth → Vivado block design")
print("→ Generate bitstream → Deploy to PYNQ-Z2 → Run pynq_package/PARETO2_PYNQ_Z2_TEST_PACKAGE/run_pareto2_pynq_test.py")
print()
print("⚠️  Until board validation is complete:")
print("  - Latency must be reported as HLS_ESTIMATED, not PYNQ_MEASURED")
print("  - Speedup vs ViT4Mal must be marked as estimated")
print("  - Power must be marked as PENDING")

---
## Section H: Summary and Final Checklist
---

In [ ]:
print("=" * 60)
print("FINAL PAPER EXPERIMENTS — CHECKLIST")
print("=" * 60)

checklist = [
    ('✅', 'Table 1: Model Accuracy Comparison', os.path.exists(f'{TABLES}/table1_model_accuracy_comparison.csv')),
    ('✅', 'Table 2: HLS Optimization Path', os.path.exists(f'{TABLES}/table2_hls_optimization_path.csv')),
    ('✅', 'Table 3: Comparison with ViT4Mal', os.path.exists(f'{TABLES}/table3_compare_with_vit4mal.csv')),
    ('✅' if os.path.exists(f'{TABLES}/table4_cpu_gpu_fpga_latency.csv') else '⚠️', 
     'Table 4: CPU/GPU/FPGA Latency', os.path.exists(f'{TABLES}/table4_cpu_gpu_fpga_latency.csv')),
    ('✅' if os.path.exists(f'{FIGURES}/fig1_acc_vs_lut.png') else '⚠️',
     'Figure 1: Accuracy vs LUT', os.path.exists(f'{FIGURES}/fig1_acc_vs_lut.png')),
    ('✅' if os.path.exists(f'{FIGURES}/fig2_acc_vs_latency.png') else '⚠️',
     'Figure 2: Accuracy vs Latency', os.path.exists(f'{FIGURES}/fig2_acc_vs_latency.png')),
    ('✅', 'PYNQ test package (script + README)', os.path.exists(f'{FP}/pynq_package/PARETO2_PYNQ_Z2_TEST_PACKAGE/run_pareto2_pynq_test.py')),
    ('✅', 'PYNQ eval samples copied', os.path.exists(f'{FP}/pynq_package/PARETO2_PYNQ_Z2_TEST_PACKAGE/eval_samples_q16.npz')),
    ('✅' if os.path.exists(f'{FP}/reports/final_experiment_status.md') else '⚠️',
     'Final experiment status report', os.path.exists(f'{FP}/reports/final_experiment_status.md')),
]

for emoji, item, exists in checklist:
    status = 'DONE' if exists else 'PENDING'
    print(f"  {emoji} {item:50s} → {status}")

print()
print("--- CAN CLAIM IN PAPER ---")
claims = [
    'Q16 PARETO2 fits PYNQ-Z2 by HLS resource estimation',
    'HLS estimated latency = 0.74084 ms at 100MHz (NOT board-measured)',
    'Accuracy on eval200 = 94.5% (hw_label_accuracy from HLS CSIM)',
    'Accuracy on full test = 94.19% (Q32 software baseline)',
    'Q16 fake quant accuracy = 95.5% on eval200',
    'DSP reduced from 2,410 → 156 by removing UNROLL + using ALLOCATION pragmas',
    '1,093× estimated speedup vs ViT4Mal fastest (HLS estimated, pending board validation)',
]
for c in claims:
    print(f"  ✅ {c}")

print()
print("--- CANNOT CLAIM YET ---")
cannot = [
    'Board-level measured latency or speedup over ViT4Mal',
    'FPGA power consumption (no Vivado power report yet)',
    'Sparse attention results (not yet implemented in HLS)',
    'Event-sparse attention results (not yet implemented)',
    '128×128 RGB ViT4Mal-aligned results (our model uses 32×32 grayscale)',
    'Final FPGA throughput in samples/sec (requires board measurement)',
]
for c in cannot:
    print(f"  ❌ {c}")

print()
print("All done. Final paper experiment materials generated.")

---
## Next Steps

1. **Run Section E in Docker Jupyter** to get CPU/ONNX latency benchmarks → updates Table 4
2. **Run Vivado IP export + block design** from PARETO2 csynth results
3. **Generate .bit and .hwh** → copy to PYNQ package
4. **Deploy to PYNQ-Z2** → run `run_pareto2_pynq_test.py` → fills all PENDING_BOARD values
5. **Implement sparse attention HLS** → add rows to Table 2 and Pareto figures

---
*Notebook generated from Docker Jupyter kernel. Do not run on host conda environment.*